In [1]:
import json
data = []

with open(
    r"C:\Users\Hari\Desktop\Projects\AI\India runs 2026\[PUB] India_runs_data_and_ai_challenge\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\candidates.jsonl",
    "r",
    encoding="utf-8"
) as file:

    for line in file:
        data.append(json.loads(line))



In [2]:
len(data)

100000

In [3]:
print(data[0].keys())

dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])


In [4]:
candidate = data[0]

print(candidate["profile"])
print(candidate["skills"])

{'anonymized_name': 'Ira Vora', 'headline': 'Backend Engineer | SQL, Spark, Cloud', 'summary': "Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.", 'location': 'Toronto', 'country': 'Canada', 'years_of_experience': 6.9, 'current_title': 'Backend Engineer', 'current_company': 'Mindtree', 'current_company_size': '10001+', 'current_industry': 'IT Services'}
[{'name': 'Tailwind', 'proficiency': 'intermediate', 'endo

In [5]:
candidate = data[0]

document = f"""
{candidate['profile']['headline']}

{candidate['profile']['summary']}

Skills:
{', '.join(skill['name'] for skill in candidate['skills'])}
"""
print(document)


Backend Engineer | SQL, Spark, Cloud

Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice.

Skills:
Tailwind, NLP, Image Classification, Fine-tuning LLMs, Weights & Biases, Speech Recognition, Photoshop, TTS, LoRA, Apache Beam, AWS, Flask, BentoML, Milvus, GANs, Statistical Modeling, GCP



In [6]:
documents = []

for candidate in data:
    document = f"""
{candidate['profile']['headline']}

{candidate['profile']['summary']}

Skills:
{', '.join(skill['name'] for skill in candidate['skills'])}
"""
    documents.append(document)

In [7]:
len(documents)

100000

In [8]:
import torch
import torchvision  # Just to verify it imports with no errors now!
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
from docx import Document

doc = Document("C:\\Users\\Hari\\Desktop\\Projects\\AI\\India runs 2026\\[PUB] India_runs_data_and_ai_challenge\\[PUB] India_runs_data_and_ai_challenge\\India_runs_data_and_ai_challenge\\job_description.docx")
text = "\n".join([p.text for p in doc.paragraphs])
lengths = [len(doc) for doc in documents[:100]]

print("avg:", sum(lengths)/len(lengths))
print("max:", max(lengths))
print("min:", min(lengths))


avg: 678.6
max: 1070
min: 546


In [10]:
print(len(documents))

100000


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

# jd_embedding = model.encode([text], device="cuda")

# docs_embeddings = model.encode(
#     documents,
#     batch_size=256,  # Increased from 64
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     device="cuda"
# )


In [12]:
import numpy as np
docs_embeddings = np.load("candidate_embeddings.npy")

In [13]:
jd_embedding = model.encode([text])

In [14]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(jd_embedding, docs_embeddings)
print(similarities)

[[0.726144   0.7510261  0.7549842  ... 0.7490512  0.73288053 0.78112024]]


In [15]:
similarities.shape

(1, 100000)

In [16]:
number = 100000

In [17]:
top_500 = np.argsort(similarities[0])[::-1][:number]


In [18]:
def candidate_score(candidate, similarity):

    score = similarity * 100

    exp = candidate["profile"]["years_of_experience"]

    if 5 <= exp <= 9:
        score += 10

    if candidate["redrob_signals"]["open_to_work_flag"]:
        score += 5

    score += (
        candidate["redrob_signals"]["recruiter_response_rate"]
        * 3
    )

    return score

In [19]:
results = []

for idx in top_500:

    candidate = data[idx]

    score = candidate_score(
        candidate,
        similarities[0][idx]
    )

    results.append((idx, score))

results.sort(
    key=lambda x: x[1],
    reverse=True
)

In [37]:
def domain_validation_score(candidate):

    score = 0
    reasons = []

    # -------------------------------
    # Build one text from title + career
    # -------------------------------

    title = (
        candidate["profile"]["current_title"]
        + " "
        + candidate["profile"]["headline"]
    ).lower()

    career = " ".join(
        job["title"] + " " + job["description"]
        for job in candidate["career_history"]
    ).lower()

    skills = " ".join(
        skill["name"]
        for skill in candidate["skills"]
    ).lower()

    text = title + " " + career + " " + skills

    # -------------------------------
    # Positive Signals
    # -------------------------------

    good_keywords = {
        "machine learning": 8,
        "ml": 4,
        "ai engineer": 8,
        "artificial intelligence": 8,
        "nlp": 8,
        "search": 6,
        "retrieval": 8,
        "ranking": 8,
        "recommendation": 8,
        "vector": 5,
        "embedding": 5,
        "llm": 6,
        "rag": 6,
        "transformer": 4,
        "sentence transformer": 5,
        "faiss": 6,
        "pinecone": 6,
        "qdrant": 6,
        "weaviate": 6,
        "milvus": 6,
        "bm25": 6,
        "cross encoder": 6,
        "information retrieval": 8
    }

    for keyword, weight in good_keywords.items():
        if keyword in text:
            score += weight

    # -------------------------------
    # Negative Signals
    # -------------------------------

    bad_keywords = {
        "mechanical engineer": -35,
        "civil engineer": -35,
        "customer support": -30,
        "sales executive": -30,
        "marketing": -25,
        "content writer": -25,
        "accountant": -30,
        "hr manager": -30,
        "operations manager": -20,
        "business analyst": -15,
        "project manager": -20,
    "graphic designer": -30,
    "ui designer": -30,
    "ux designer": -30,
    }

    for keyword, penalty in bad_keywords.items():
        if keyword in title:
            score += penalty
            reasons.append(keyword)

    return score, reasons

In [21]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [22]:
top_100 = results

In [23]:
pairs = []

for idx, score in top_100:

    candidate = data[idx]

    candidate_text = f"""
    {candidate['profile']['headline']}
    {candidate['profile']['summary']}
    """

    pairs.append(
        [text, candidate_text]  # text = JD
    )

In [24]:
import bm25s

corpus = bm25s.tokenize(documents)

bm25 = bm25s.BM25()
bm25.index(corpus)

query = bm25s.tokenize(text)

results500, scores500 = bm25.retrieve(
    query,
    corpus=documents,   # <-- THIS IS MISSING
    k=500
)



Split strings:   0%|          | 0/100000 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/100000 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/100000 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

In [25]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(jd_embedding, docs_embeddings)

In [26]:

top_indices = np.argsort(
    similarities[0]
)[::-1][:number]

top_scores = similarities[0][top_indices]

In [27]:
vector_top_500_indices = np.argsort(similarities[0])[::-1][:number].tolist()
# Create mapping: document -> original candidate index
doc_to_index = {
    doc: idx
    for idx, doc in enumerate(documents)
}

# Convert BM25 retrieved documents back to candidate indices
bm25_top_500_indices = [
    doc_to_index[doc]
    for doc in results500[0]
]

In [ ]:
def run_rrf(list1, list2, k=60):
    rrf_scores = {}
    
    # Process first list (Vector)
    for rank, idx in enumerate(list1, start=1):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k + rank))
        
        
    # Process second list (BM25)
    for rank, idx in enumerate(list2, start=1):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k + rank))
        
    # Sort candidate indices by their RRF score descending
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_rrf

combined_rrf_results = run_rrf(vector_top_500_indices, bm25_top_500_indices, k=60)

In [29]:
top_100 = combined_rrf_results

In [30]:
pairs = []

for idx, score in top_100:

    candidate = data[idx]

    candidate_text = f"""
        Title:
        {candidate['profile']['headline']}

        Summary:
        {candidate['profile']['summary']}

        Experience:
        {candidate['profile']['years_of_experience']} years

        Skills:
        {', '.join(skill['name'] for skill in candidate['skills'])}

        Career:
        {' '.join(job['description'] for job in candidate['career_history'])}
    """

    pairs.append(
        [text, candidate_text]  # text = JD
    )

In [32]:
# cross_scores = cross_encoder.predict(
#     pairs,
#     batch_size=32
# )
cross_scores = np.load("cross_encoder_scores.npy")


In [38]:
from scipy.special import expit



final_results = []

for (idx, old_score), cross_score in zip(top_100, cross_scores):
    normalized_cross = expit(cross_score) * 100
    
    candidate = data[idx]

    domain_score, reasons = domain_validation_score(candidate)

    heuristic_score = candidate_score(
        candidate,
        similarities[0][idx]
    )

    final_score = (
        normalized_cross * 0.6 +
        heuristic_score * 0.4 +
        domain_score
    )

    final_results.append(
        (idx, final_score, reasons)
    )

final_results.sort(
    key=lambda x: x[1],
    reverse=True
)

In [39]:
for idx, score, reasons in final_results[:100]:

    c = data[idx]

    print(
        c["candidate_id"],
        c["profile"]["headline"],
        score,
        reasons
    )

CAND_0077337 Staff Machine Learning Engineer | Building AI-native search & ranking systems 153.85513 []
CAND_0033861 Senior NLP Engineer | Building AI-native search & ranking systems 153.143 []
CAND_0039754 Senior Applied Scientist | Building AI-native search & ranking systems 145.58551 []
CAND_0071974 Senior AI Engineer | Production ML at scale | 7.8+ yrs 141.95319 []
CAND_0042506 Search Engineer | Search, Ranking & Retrieval 141.88702 []
CAND_0041669 Recommendation Systems Engineer | Search, Ranking & Retrieval 141.82022 []
CAND_0046525 Senior Machine Learning Engineer | Building AI-native search & ranking systems 139.01578 []
CAND_0018499 Senior Machine Learning Engineer | Building AI-native search & ranking systems 138.91473 []
CAND_0081846 Lead AI Engineer | LLMs, RAG, Vector Search | ex-Top Tech 138.80121 []
CAND_0044855 Senior Data Scientist | ML, NLP, Recommendation Systems 138.74649 []
CAND_0055992 AI Engineer | Search, Ranking & Retrieval 138.31583 []
CAND_0093547 Senior Mach

In [35]:
class CandidateExplainer:
    def __init__(self, data, documents, similarities):
        self.data = data
        self.documents = documents
        self.similarities = similarities
        
    def generate_explanation(self, idx, final_score, cross_score, rrf_score, bm25_rank=None):
        candidate = self.data[idx]
        profile = candidate["profile"]
        signals = candidate["redrob_signals"]
        
        # 1. Calculate Component Strengths (Normalized for human reading)
        vector_sim = float(self.similarities[0][idx])
        
        # 2. Extract Top matching keywords (Intersection of skills and JD keywords)
        # Simple heuristic: what skills do they have that contribute to their match
        skills = [s["name"] for s in candidate["skills"]]
        
        # 3. Formulate the core reasons
        reasons = []
        if cross_score > 50:  # Changed from 0.5 because it's now a 0-100 score
            reasons.append(f"Strong contextual alignment between background and JD (Cross-Encoder: {cross_score:.1f}%)")
        else:
            reasons.append(f"Moderate contextual fit with specific role nuances (Cross-Encoder: {cross_score:.1f}%)")

        if 5 <= profile["years_of_experience"] <= 9:
            reasons.append(f"Ideal experience sweet-spot ({profile['years_of_experience']} years)")
        else:
            reasons.append(f"Experience level: {profile['years_of_experience']} years")
            
        if signals["open_to_work_flag"]:
            reasons.append("Actively looking ('Open to Work' status active)")
            
        if signals["recruiter_response_rate"] > 0.8:
            reasons.append(f"Highly responsive candidate (Response rate: {signals['recruiter_response_rate']*100:.0f}%)")
            
        if signals["github_activity_score"] > 70:
            reasons.append(f"Strong open-source/GitHub presence (Activity Score: {signals['github_activity_score']})")

        # 4. Construct Structured Output
        explanation = {
            "candidate_id": candidate["candidate_id"],
            "headline": profile["headline"],
            "final_composite_score": round(final_score, 2),
            "score_breakdown": {
                "Semantic Alignment (Cross-Encoder)": f"{cross_score:.4f}",
                "Vector Semantic Match (Bi-Encoder)": f"{vector_sim:.4f}",
                "RRF Fusion Score": f"{rrf_score:.5f}",
                "Experience & Signal Tier": "High Fit" if final_score > 5.0 else "Regular Fit"
            },
            "key_skills": skills[:6],
            "justification_bullets": reasons,
            "logistics": {
                "Notice Period": f"{signals['notice_period_days']} days",
                "Work Mode": signals["preferred_work_mode"],
                "Expected Salary": f"{signals['expected_salary_range_inr_lpa']['min']}-{signals['expected_salary_range_inr_lpa']['max']} LPA"
            }
        }
        return explanation

    def print_report(self, explanation, rank=1):
        print(f"=== [RANK {rank}] Candidate Profile Explanation ===")
        print(f"ID: {explanation['candidate_id']} | {explanation['headline']}")
        print(f"Final Combined Score: {explanation['final_composite_score']}")
        print("-" * 50)
        print("Score Breakdown Contributors:")
        for k, v in explanation['score_breakdown'].items():
            print(f"  • {k}: {v}")
        print("\nWhy this candidate was picked:")
        for bullet in explanation['justification_bullets']:
            print(f"  ✓ {bullet}")
        print("\nLogistics & Practical Fit:")
        for k, v in explanation['logistics'].items():
            print(f"  • {k}: {v}")
        print(f"Key Identified Skills: {', '.join(explanation['key_skills'])}")
        print("=" * 60 + "\n")

In [ ]:
# Initialize the explainer engine
explainer = CandidateExplainer(data, documents, similarities)

# Re-run your final compilation step but save the interim scores for the explainer
final_results_with_metadata = []

for i, ((idx, rrf_score), cross_score) in enumerate(zip(top_100, cross_scores)):
    candidate = data[idx]
    
    # Your existing scoring function logic
    normalized_cross = expit(cross_score) * 100

    heuristic_score = candidate_score(
    candidate,
    similarities[0][idx]
    )

    domain_score, reasons = domain_validation_score(candidate)

    final_score = (
        normalized_cross * 0.6 +
        heuristic_score * 0.4 +
        domain_score
    )
    final_results_with_metadata.append({
        "idx": idx,
        "final_score": final_score,
        "cross_score": cross_score,
        "rrf_score": rrf_score
    })

# Sort by final score descending
final_results_with_metadata.sort(key=lambda x: x["final_score"], reverse=True)

# Generate and print explanations for your top 5 submissions
print("GENERATING EXPLAINABILITY SUBMISSION REPORT...\n")
for rank, item in enumerate(final_results_with_metadata[:5], start=1):
    exp_data = explainer.generate_explanation(
        idx=item["idx"],
        final_score=item["final_score"],
        cross_score=item["cross_score"],
        rrf_score=item["rrf_score"]
    )
    explainer.print_report(exp_data, rank=rank)

GENERATING EXPLAINABILITY SUBMISSION REPORT...

=== [RANK 1] Candidate Profile Explanation ===
ID: CAND_0046525 | Senior Machine Learning Engineer | Building AI-native search & ranking systems
Final Combined Score: 38.84000015258789
--------------------------------------------------
Score Breakdown Contributors:
  • Semantic Alignment (Cross-Encoder): -2.7779
  • Vector Semantic Match (Bi-Encoder): 0.8362
  • RRF Fusion Score: 0.02828
  • Experience & Signal Tier: High Fit

Why this candidate was picked:
  ✓ Moderate contextual fit with specific role nuances (Cross-Encoder: -2.8%)
  ✓ Ideal experience sweet-spot (6.1 years)
  ✓ Actively looking ('Open to Work' status active)
  ✓ Highly responsive candidate (Response rate: 88%)

Logistics & Practical Fit:
  • Notice Period: 60 days
  • Work Mode: onsite
  • Expected Salary: 28.4-51.9 LPA
Key Identified Skills: Elasticsearch, Redux, LangChain, Machine Learning, LlamaIndex, Information Retrieval

=== [RANK 2] Candidate Profile Explanation

In [40]:
AI_CORE_SKILLS = {
    "machine learning",
    "deep learning",
    "nlp",
    "llms",
    "rag",
    "langchain",
    "vector search",
    "faiss",
    "pinecone",
    "embeddings",
    "information retrieval",
    "recommendation systems",
    "semantic search",
    "transformers",
    "hugging face",
    "tensorflow",
    "pytorch",
    "scikit-learn",
    "opensearch",
    "elasticsearch"
}
def build_reason(candidate):

    profile = candidate["profile"]
    signals = candidate["redrob_signals"]

    skills = [
        s["name"]
        for s in candidate["skills"]
    ]

    ai_skill_count = sum(
        1
        for skill in skills
        if skill.lower() in {s.lower() for s in AI_CORE_SKILLS}
    )

    reason = (
        f"{profile['current_title']}; "
        f"{profile['years_of_experience']:.1f} yrs; "
        f"{ai_skill_count} AI core skills; "
        f"response rate {signals['recruiter_response_rate']:.2f}"
    )

    return reason


In [42]:

top_100 = final_results[:100]

In [43]:
import pandas as pd

scores = [score for _, score, _ in final_results]

min_score = min(scores)
max_score = max(scores)

rows = []

for rank, (idx, score, reasons) in enumerate(top_100, start=1):

    candidate = data[idx]

    normalized_score = (
        score - min_score
    ) / (
        max_score - min_score
    )

    rows.append({
        "candidate_id": candidate["candidate_id"],
        "rank": rank,
        "score": round(normalized_score, 4),
        "reasoning": build_reason(candidate)
    })

submission = pd.DataFrame(rows)

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head(10))

   candidate_id  rank   score  \
0  CAND_0077337     1  1.0000   
1  CAND_0033861     2  0.9955   
2  CAND_0039754     3  0.9483   
3  CAND_0071974     4  0.9256   
4  CAND_0042506     5  0.9252   
5  CAND_0041669     6  0.9248   
6  CAND_0046525     7  0.9072   
7  CAND_0018499     8  0.9066   
8  CAND_0081846     9  0.9059   
9  CAND_0044855    10  0.9056   

                                           reasoning  
0  Staff Machine Learning Engineer; 7.0 yrs; 7 AI...  
1  Senior NLP Engineer; 8.0 yrs; 7 AI core skills...  
2  Senior Applied Scientist; 16.2 yrs; 7 AI core ...  
3  Senior AI Engineer; 7.8 yrs; 6 AI core skills;...  
4  Search Engineer; 4.2 yrs; 7 AI core skills; re...  
5  Recommendation Systems Engineer; 8.0 yrs; 4 AI...  
6  Senior Machine Learning Engineer; 6.1 yrs; 7 A...  
7  Senior Machine Learning Engineer; 7.2 yrs; 8 A...  
8  Lead AI Engineer; 6.7 yrs; 10 AI core skills; ...  
9  Senior Data Scientist; 6.6 yrs; 7 AI core skil...  
